In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import spacy

print("Setup complete!")

Setup complete!


In [ ]:
ASPECT_VOCAB = {
    "battery": ["battery", "charge", "charging", "life", "drain", "power"],
    "screen":  ["screen", "display", "brightness", "resolution", "pixels"],
    "camera":  ["camera", "photo", "picture", "image", "lens", "zoom"],
    "price":   ["price", "cost", "value", "cheap", "expensive", "worth"],
    "delivery":["delivery", "shipping", "packaging", "arrived", "box"]
}

WORD_TO_ASPECT = {}
for aspect, keywords in ASPECT_VOCAB.items():
    for word in keywords:
        WORD_TO_ASPECT[word] = aspect

print("Vocabulary loaded")
print(f"Total keywords tracked: {len(WORD_TO_ASPECT)}")
print("\nWord → Aspect mapping sample:")
for word, aspect in list(WORD_TO_ASPECT.items())[:8]:
    print(f"  '{word}' → '{aspect}'")

Vocabulary loaded!
Total keywords tracked: 28

Word → Aspect mapping sample:
  'battery' → 'battery'
  'charge' → 'battery'
  'charging' → 'battery'
  'life' → 'battery'
  'drain' → 'battery'
  'power' → 'battery'
  'screen' → 'screen'
  'display' → 'screen'


In [ ]:
nlp = spacy.load("en_core_web_sm")

def extract_aspects(text):

    #Returns list of unique aspects found in text.

    doc = nlp(text.lower())
    found = set()

    # 1. Noun chunks (e.g. "the battery life")
    for chunk in doc.noun_chunks:
        for token in chunk:
            if token.text in WORD_TO_ASPECT:
                found.add(WORD_TO_ASPECT[token.text])

    # 2. individual noun tokens (catches what chunking misses)
    for token in doc:
        if token.pos_ in ("NOUN", "PROPN") and token.text in WORD_TO_ASPECT:
            found.add(WORD_TO_ASPECT[token.text])

    return list(found)

print("extract_aspects() defined!")

extract_aspects() defined!


In [ ]:
test_cases = [
    "The battery drains within 3 hours and the screen resolution is poor.",
    "Great camera and fast delivery, but a bit expensive.",
    "Display brightness is excellent for outdoor use.",
    "Charging takes forever and the box was damaged.",
    "Absolutely love this phone, works perfectly!",  
]

print("Testing aspect extraction:\n")
for text in test_cases:
    aspects = extract_aspects(text)
    print(f"Text:    {text}")
    print(f"Aspects: {aspects}")
    print()

Testing aspect extraction:

Text:    The battery drains within 3 hours and the screen resolution is poor.
Aspects: ['battery', 'screen']

Text:    Great camera and fast delivery, but a bit expensive.
Aspects: ['delivery', 'camera']

Text:    Display brightness is excellent for outdoor use.
Aspects: ['screen']

Text:    Charging takes forever and the box was damaged.
Aspects: ['delivery']

Text:    Absolutely love this phone, works perfectly!
Aspects: []



In [ ]:
from datasets import load_dataset

print("Loading a small sample to test on real reviews")

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    trust_remote_code=True,
    streaming=True
)

sample_records = []
for i, row in enumerate(dataset['full']):
    if i >= 200:
        break
    sample_records.append({'text': row['text'], 'rating': row['rating']})

sample_df = pd.DataFrame(sample_records).dropna()

print("\nAspect extraction on 10 real reviews:\n")
for _, row in sample_df.head(10).iterrows():
    aspects = extract_aspects(str(row['text']))
    if aspects:  # only print if aspects found
        print(f"Rating: {row['rating']} | Aspects: {aspects}")
        print(f"Review: {str(row['text'])[:120]}...")
        print()

d:\Project 2\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading a small sample to test on real reviews...

Aspect extraction on 10 real reviews:

Rating: 1.0 | Aspects: ['delivery', 'camera']
Review: These didn’t work. Idk if they were damaged in shipping or what, but the lenses were loose or something. I could see hal...

Rating: 5.0 | Aspects: ['price']
Review: I've bought these headphones three times because I love them so much and overuse them and find ways to break the plug to...

Rating: 5.0 | Aspects: ['battery']
Review: Light weight, quiet and totally awesome!!! It doesn’t come with a charging base, but the cord ends in a USB so you can p...

Rating: 5.0 | Aspects: ['price']
Review: pretty good for the price....



In [6]:
# How many reviews mention at least one aspect?
sample_df['aspects'] = sample_df['text'].apply(
    lambda x: extract_aspects(str(x))
)
sample_df['aspect_count'] = sample_df['aspects'].apply(len)

coverage = (sample_df['aspect_count'] > 0).mean() * 100
print(f"Reviews with at least one aspect detected: {coverage:.1f}%")
print(f"\nMost common aspects found:")
from collections import Counter
all_aspects = [a for aspects in sample_df['aspects'] for a in aspects]
for aspect, count in Counter(all_aspects).most_common():
    print(f"  {aspect}: {count} mentions")

Reviews with at least one aspect detected: 26.5%

Most common aspects found:
  price: 20 mentions
  camera: 17 mentions
  battery: 13 mentions
  screen: 9 mentions
  delivery: 6 mentions


In [ ]:
from transformers import pipeline as hf_pipeline

zero_shot = hf_pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def extract_aspects_zeroshot(text, threshold=0.3):
    #Fallback using BART when rule-based finds nothing.
    candidate_labels = list(ASPECT_VOCAB.keys())
    result = zero_shot(text, candidate_labels, multi_label=True)
    return [
        label for label, score
        in zip(result['labels'], result['scores'])
        if score > threshold
    ]

def extract_aspects_combined(text):
    #Rule-based first, BART fallback if nothing found.
    aspects = extract_aspects(text)
    if not aspects:
        aspects = extract_aspects_zeroshot(text)
    return aspects

test = "The zoom capability is impressive but it costs too much."
print("Rule-based:", extract_aspects(test))
print("Combined:  ", extract_aspects_combined(test))

d:\Project 2\.venv1\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vyom\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 9320.76it/s]


Rule-based: ['camera']
Combined:   ['camera']
